In [0]:
%sql
TRUNCATE TABLE workspace.silver.dim_customer_tmp;

INSERT INTO workspace.silver.dim_customer_tmp (
    SoldToParty,
    CustomerGroup,
    CustomerAccountAssignmentGroup,
    CustomerPaymentTerms,
    _rescued_data,
    _ingestion_timestamp,
    Status_Cleansing,
    Status_DQ1,
    Status_DQ2,
    Status_DQ3,
    Status_DQ4,
    Status_DQ5,
    Status_Process
)
SELECT DISTINCT
    SoldToParty,
    CustomerGroup,
    CustomerAccountAssignmentGroup,
    CustomerPaymentTerms,
    _rescued_data,
    _ingestion_timestamp,
    'PENDING' AS Status_Cleansing,
    'PENDING' AS Status_DQ1,
    'PENDING' AS Status_DQ2,
    'PENDING' AS Status_DQ3,
    'PENDING' AS Status_DQ4,
    'PENDING' AS Status_DQ5,
    'IN_PROGRESS' AS Status_Process
FROM workspace.bronze.sales_order_header;

In [0]:
%sql
UPDATE workspace.silver.dim_customer_tmp
SET 
    SoldToParty = CASE 
        WHEN regexp_replace(SoldToParty, '^0+', '') = '' AND SoldToParty IS NOT NULL THEN '0'
        ELSE coalesce(regexp_replace(SoldToParty, '^0+', ''), SoldToParty)
    END,
    CustomerGroup = trim(upper(CustomerGroup)),
    CustomerAccountAssignmentGroup = trim(upper(CustomerAccountAssignmentGroup)),
    CustomerPaymentTerms = trim(upper(CustomerPaymentTerms)),
    Status_Cleansing = 'COMPLETED'
WHERE Status_Cleansing = 'PENDING' 
  AND Status_Process = 'IN_PROGRESS';

DQ1 Rule (Mandatory / Not NULL) UPDATE only.

DQ2 Rule (Customer Uniqueness / Batch-level Integrity) MERGE only.

DQ3 Rule (Clean Format / _rescued_data IS NULL) UPDATE only.

DQ4 Rule (SAP Standard Domain and Field Lengths) UPDATE only.

DQ5 Rule (Valid Characters in Key) UPDATE only.

The Quarantine Sweeper: A single INSERT that collects all accumulated failures.

Final Promotion: MERGE approved records into dim_customer_stg

In [0]:
%sql
UPDATE workspace.silver.dim_customer_tmp
SET 
    Status_DQ1 = CASE 
        WHEN SoldToParty IS NULL OR trim(SoldToParty) = '' THEN 'FAILED'
        ELSE 'PASSED'
    END,
    Status_Process = CASE 
        WHEN SoldToParty IS NULL OR trim(SoldToParty) = '' THEN 'QUARANTINED'
        ELSE Status_Process
    END
WHERE Status_Cleansing = 'COMPLETED' 
  AND Status_Process = 'IN_PROGRESS';

In [0]:
%sql
MERGE INTO workspace.silver.dim_customer_tmp AS tgt
USING (
    SELECT 
        SoldToParty,
        COUNT(DISTINCT named_struct(
            'CustomerGroup', CustomerGroup,
            'CustomerAccountAssignmentGroup', CustomerAccountAssignmentGroup,
            'CustomerPaymentTerms', CustomerPaymentTerms
        )) AS conflict_count
    FROM workspace.silver.dim_customer_tmp
    WHERE Status_Process = 'IN_PROGRESS'
    GROUP BY SoldToParty
) AS src
ON tgt.SoldToParty = src.SoldToParty
WHEN MATCHED AND tgt.Status_Process = 'IN_PROGRESS' AND tgt.Status_DQ1 = 'PASSED' THEN
    UPDATE SET 
        tgt.Status_DQ2 = CASE 
            WHEN src.conflict_count > 1 THEN 'FAILED' 
            ELSE 'PASSED' 
        END,
        tgt.Status_Process = CASE 
            WHEN src.conflict_count > 1 THEN 'QUARANTINED' 
            ELSE tgt.Status_Process 
        END;

In [0]:
%sql
UPDATE workspace.silver.dim_customer_tmp
SET 
    Status_DQ3 = CASE 
        WHEN SoldToParty LIKE '0%' AND SoldToParty <> '0' THEN 'FAILED'
        ELSE 'PASSED'
    END,
    Status_Process = CASE 
        WHEN SoldToParty LIKE '0%' AND SoldToParty <> '0' THEN 'QUARANTINED'
        ELSE Status_Process
    END
WHERE Status_Process = 'IN_PROGRESS'
  AND Status_DQ2 = 'PASSED';

In [0]:
%sql
UPDATE workspace.silver.dim_customer_tmp
SET 
    Status_DQ4 = CASE 
        WHEN length(coalesce(CustomerGroup, '')) > 4 
          OR length(coalesce(CustomerAccountAssignmentGroup, '')) > 4 
          OR length(coalesce(CustomerPaymentTerms, '')) > 4 THEN 'FAILED'
        ELSE 'PASSED'
    END,
    Status_Process = CASE 
        WHEN length(coalesce(CustomerGroup, '')) > 4 
          OR length(coalesce(CustomerAccountAssignmentGroup, '')) > 4 
          OR length(coalesce(CustomerPaymentTerms, '')) > 4 THEN 'QUARANTINED'
        ELSE Status_Process
    END
WHERE Status_Process = 'IN_PROGRESS'
  AND Status_DQ3 = 'PASSED';

In [0]:
%sql
UPDATE workspace.silver.dim_customer_tmp
SET 
    Status_DQ5 = CASE 
        WHEN _rescued_data IS NOT NULL THEN 'FAILED'
        ELSE 'PASSED'
    END,
    Status_Process = CASE 
        WHEN _rescued_data IS NOT NULL THEN 'QUARANTINED'
        ELSE Status_Process
    END
WHERE Status_Process = 'IN_PROGRESS'
  AND Status_DQ4 = 'PASSED';

In [0]:
%sql
-- Sweeper: Consolidated routing of rejected records to Quarantine
INSERT INTO workspace.silver.sales_order_quarantine (
    SourceEntity,
    RecordIdentifier,
    FailedRule,
    FailureSeverity,
    RawRecord,
    IngestionTimestamp
)
SELECT 
    'DIM_CUSTOMER' AS SourceEntity,
    coalesce(SoldToParty, 'UNKNOWN_KEY') AS RecordIdentifier,
    CASE 
        WHEN Status_DQ1 = 'FAILED' THEN 'DQ1_NOT_NULL_VIOLATION'
        WHEN Status_DQ2 = 'FAILED' THEN 'DQ2_ATTRIBUTE_CONFLICT'
        WHEN Status_DQ3 = 'FAILED' THEN 'DQ3_ALPHA_CONVERSION_ERROR'
        WHEN Status_DQ4 = 'FAILED' THEN 'DQ4_DOMAIN_LENGTH_EXCEEDED'
        WHEN Status_DQ5 = 'FAILED' THEN 'DQ5_RESCUED_DATA_CORRUPT'
        ELSE 'UNKNOWN_FAILURE'
    END AS FailedRule,
    'HARD' AS FailureSeverity,
    to_json(named_struct(
        'SoldToParty', SoldToParty,
        'CustomerGroup', CustomerGroup,
        'CustomerAccountAssignmentGroup', CustomerAccountAssignmentGroup,
        'CustomerPaymentTerms', CustomerPaymentTerms,
        '_rescued_data', _rescued_data
    )) AS RawRecord,
    current_timestamp() AS IngestionTimestamp
FROM workspace.silver.dim_customer_tmp
WHERE Status_Process = 'QUARANTINED';

In [0]:
%sql
UPDATE workspace.silver.dim_customer_tmp
SET Status_Process = 'READY_FOR_STG'
WHERE Status_Process = 'IN_PROGRESS'
  AND Status_DQ1 = 'PASSED'
  AND Status_DQ2 = 'PASSED'
  AND Status_DQ3 = 'PASSED'
  AND Status_DQ4 = 'PASSED'
  AND Status_DQ5 = 'PASSED';

In [0]:
%sql
-- Promotion: Load and idempotency into target dimension
MERGE INTO workspace.silver.dim_customer_stg AS tgt
USING (
    SELECT 
        SoldToParty,
        CustomerGroup,
        CustomerAccountAssignmentGroup,
        CustomerPaymentTerms,
        min(_ingestion_timestamp) AS ValidFrom
    FROM workspace.silver.dim_customer_tmp
    WHERE Status_Process = 'READY_FOR_STG'
    GROUP BY 
        SoldToParty,
        CustomerGroup,
        CustomerAccountAssignmentGroup,
        CustomerPaymentTerms
) AS src
ON tgt.SoldToParty = src.SoldToParty AND tgt.IsCurrent = TRUE
WHEN MATCHED AND (
    tgt.CustomerGroup <=> src.CustomerGroup = FALSE OR
    tgt.CustomerAccountAssignmentGroup <=> src.CustomerAccountAssignmentGroup = FALSE OR
    tgt.CustomerPaymentTerms <=> src.CustomerPaymentTerms = FALSE
) THEN
    -- If commercial attributes changed, close previous version
    UPDATE SET 
        tgt.ValidTo = current_timestamp(),
        tgt.IsCurrent = FALSE
WHEN NOT MATCHED THEN
    -- Insert new customer
    INSERT (
        SoldToParty,
        CustomerGroup,
        CustomerAccountAssignmentGroup,
        CustomerPaymentTerms,
        ValidFrom,
        ValidTo,
        IsCurrent
    )
    VALUES (
        src.SoldToParty,
        src.CustomerGroup,
        src.CustomerAccountAssignmentGroup,
        src.CustomerPaymentTerms,
        coalesce(src.ValidFrom, current_timestamp()),
        NULL,
        TRUE
    );